# 뉴스 크롤링 데이터 노이즈 제거 (조선일보 계열)

이 노트북은 `기사 본문 전체` 컬럼에 섞여 들어간 **광고, 내비게이션 메뉴, 추천기사 목록, 댓글(100자평), 저작권 문구, 기자 약력, 동영상 플레이어 UI 텍스트** 등을
제거하고 **순수한 기사 본문만** 남기는 파이프라인입니다.

## 왜 이렇게 지저분한가?
크롤러가 페이지의 DOM을 재귀적으로 순회하면서 같은 블록(제목, 바이라인, 추천기사 등)을 **여러 번 중복 수집**했고,
그 사이사이에 내비게이션 바, 영상 플레이어 자막 설정 UI, 타부라(Taboola) 광고, '많이 본 뉴스' 목록, 댓글창, 푸터(저작권/사이트맵)가
그대로 텍스트로 섞여 들어가 있습니다. (원본 평균 길이 약 15,300자 → 실제 기사 본문은 대개 1,000자 내외)

## 정제 전략 (규칙 기반 + 선택적 LLM 보정)
1. **PREFIX 컷**: 기사 앞부분의 내비게이션/공유버튼/중복 헤드라인을 `입력 YYYY.MM.DD HH:MM` 바이라인 타임스탬프가 처음 등장하는 지점까지 제거
2. **SUFFIX 컷**: `Video Player is loading`, `100자평`, `많이 본 뉴스`, `By Taboola`, `저작권자`, `Copyright` 등 알려진 노이즈 시작 마커 중 가장 먼저 나오는 지점부터 끝까지 제거
3. **문장 중복 제거**: 스크래퍼가 중복 수집한 동일 문장을 처음 등장한 순서만 남기고 제거
4. **기자 약력/서명 제거**: 말미에 남는 '~습니다'체 자기소개, '이름+기자+부서명' 서명을 문맥(인용부호+발화귀속 표현 여부)을 보고 판별해 제거
5. **LLM 2차 정제**: 위 규칙으로도 못 잡는 잔여 노이즈(추천기사 스니펫 등)를 Claude API로 한 번 더 검수/정제

전체 2,706개 기사에 실제로 테스트하며 규칙을 다듬었습니다.

## 1. 라이브러리 설치 및 데이터 업로드

In [ ]:
# 코랩 환경 기준. 로컬 주피터라면 이 셀은 건너뛰어도 됩니다.
import pandas as pd
import re

from google.colab import files
uploaded = files.upload()  # CSV 파일을 선택해서 업로드하세요
csv_filename = list(uploaded.keys())[0]
print("업로드된 파일:", csv_filename)


In [ ]:
df = pd.read_csv(csv_filename)
print(df.shape)
print(df.columns.tolist())
df.head(2)


## 2. 정제 규칙 라이브러리

아래 함수들이 이번 정제 로직의 핵심입니다. **조선일보 계열(www.chosun.com, biz.chosun.com,
health.chosun.com, economychosun.com, news.chosun.com, realty.chosun.com) 데이터에 맞춰
튜닝**되어 있습니다. 다른 언론사 데이터가 섞여 있다면 `SUFFIX_MARKERS` 목록에 해당 매체의
푸터/광고 문구를 추가해주면 됩니다.

In [ ]:
# -*- coding: utf-8 -*-
"""
조선일보 계열(www.chosun.com, biz.chosun.com, health.chosun.com,
economychosun.com, news.chosun.com, realty.chosun.com 등) 뉴스 크롤링 데이터에서
광고/내비게이션/추천기사/댓글/저작권 문구 등 '기사 본문이 아닌' 노이즈를 제거하는 유틸.

전략
----
1) PREFIX_CUT   : 기사 앞부분의 내비게이션 메뉴, 공유버튼, 중복 헤드라인/바이라인을
                   '입력 YYYY.MM.DD HH:MM' 형태의 타임스탬프가 (제한된 앞부분 윈도우 내에서)
                   처음 등장하는 지점까지 잘라낸다.
2) SUFFIX_CUT   : 본문 뒤에 붙는 추천기사/광고/댓글/저작권 블록을 알려진 마커 문구 중
                   가장 먼저 등장하는 지점부터 끝까지 잘라낸다.
3) DEDUP        : 스크래퍼가 DOM을 재귀적으로 순회하며 만든 '문장 반복' 노이즈를
                   문장 단위로 중복 제거한다 (처음 등장한 순서만 유지).
4) SAFETY NET   : 그래도 남아있는 대표적 잔여 노이즈 문구(저작권, 이메일 서명 등)를
                   정규식으로 한 번 더 제거한다.
"""
import re

# ---------------------------------------------------------------
# 1. 접미(suffix) 노이즈 시작 마커: 이 중 하나라도 등장하면 그 지점부터 끝까지 제거
# ---------------------------------------------------------------
SUFFIX_MARKERS = [
    r"Video Player is loading",
    r"100자평",
    r"AI\s*추천\s*오늘의\s*멤버십",
    r"By Taboola",
    r"당신이\s*좋아할\s*만한\s*콘텐츠",
    r"지금\s*뜨는\s*콘텐츠",
    r"많이\s*본\s*뉴스",
    r"오늘의\s*핫뉴스",
    r"오피니언\s*사설\s*칼럼",
    r"저작권자\s*[ⓒ©]",
    r"Copyright\s*[ⓒ©]?\s*조선일보",
    r"무단\s*전재\s*및\s*재배포\s*금지",
    r"기사\s*전체보기",
    r"이\s*기사와\s*관련기사",
    r"구독수\s*\d+",
    r"제휴안내\s*구독신청",
    r"회사소개\s*기자채용",
    r"English\s*기사보기",
    r"#[가-힣A-Za-z0-9]+(?:\s+#?[가-힣A-Za-z0-9]+){1,8}\s+더보기",  # '#태그1 태그2 ... 더보기' (해시태그 + 관련기사 목록 시작)
    r"arrow_forward_ios",
    r"Powered by GliaStudios",
    r"CANCEL\s+NEXT\s+VIDEO",
    r"\[custom_chain\]",
]
SUFFIX_RE = re.compile("|".join(SUFFIX_MARKERS))

# 기자 바이라인/약력 문장에서 흔히 쓰이는 1인칭 종결어미('-습니다')와
# '이름 + 기자' 로 시작하는 소개 문장을 걸러내기 위한 패턴.
# 실제 기사 본문은 '-다/-했다' 류로 끝나는 보도체이고, '-습니다'는 인용부호 없이 등장하면
# 대개 기자 소개(약력) 문장이므로 안전하게 제거 대상으로 판단한다.
BIO_ENDING_RE = re.compile(r"습니다\.?$")
BIO_NAME_START_RE = re.compile(r"^[가-힣]{2,4}\s?기자\b")
BIO_NAME_ANY_RE = re.compile(r"(?:^|[\s#])[가-힣]{2,4}\s?기자\s")
QUOTE_CHARS = ('"', "'", '“', '”', '‘', '’')
# 인용부호가 있어도 '~라고 했다/말했다/밝혔다' 등 화자 발화 귀속 표현이 뒤따르면
# 실제 취재원 발언 인용문으로 보고 보호(제거하지 않음)한다.
ATTRIBUTION_RE = re.compile(r"(라고|며)\s*[^.]{0,10}(말했|밝혔|전했|강조했|덧붙였|했다)")

# ---------------------------------------------------------------
# 2. 접두(prefix) 컷 기준: 기사 앞부분에서 등장하는 '입력 YYYY.MM.DD HH:MM' 류 타임스탬프
#    (기자 바이라인 / 등록시각) 패턴. 이 중 앞부분 윈도우 내 마지막 매치 지점까지를 잘라낸다.
# ---------------------------------------------------------------
BYLINE_TS_RE = re.compile(
    r"입력\s*:?\s*\d{4}[.\-]\d{2}[.\-]\d{2}\.?\s*\d{1,2}:\d{2}"   # 입력 2025.02.12. 00:32 / 입력 : 2025.10.29 15:33
    r"(?:\s*\|?\s*(?:수정|업데이트)\s*:?\s*\d{4}[.\-]\d{2}[.\-]\d{2}\.?\s*\d{1,2}:\d{2})?"  # 뒤따르는 수정/업데이트 시각(옵션)
)
PREFIX_WINDOW = 4000  # 이 범위 안에서만 '기사 시작 지점'을 탐색 (본문이 그 이후에 나온다고 가정)

# ---------------------------------------------------------------
# 3. 문장 분리 (마침표/물음표/느낌표 뒤 공백 기준, 한국어 텍스트에 적합한 단순 규칙)
# ---------------------------------------------------------------
SENT_SPLIT_RE = re.compile(r'(?<=[.!?…”"\'])\s+')

# 잔여 안전망: 어디에 있든 통째로 제거할 확실한 잔여 노이즈 문구
RESIDUAL_PATTERNS = [
    re.compile(r"Copyright\s*[ⓒ©]?\s*조선일보.*?무단\s*전재\s*및\s*재배포\s*금지\.?"),
    re.compile(r"/[a-zA-Z0-9_.+-]+@chosun\.com"),          # 기자 이메일 서명
    re.compile(r"저작권자\s*[ⓒ©].*?재배포\s*금지"),
    re.compile(r"\bclose\s+Advertisements\b", re.IGNORECASE),
]

# 문단 맨 끝에 덩그러니 남은 해시태그 태그 나열(예: '#특검 #윤석열') 제거용
TRAILING_HASHTAGS_RE = re.compile(r"(?:\s*#[가-힣A-Za-z0-9]+)+\s*$")


def _cut_prefix(text: str) -> str:
    window = text[:PREFIX_WINDOW]
    m = BYLINE_TS_RE.search(window)
    if not m:
        return text
    return text[m.end():]


def _cut_suffix(text: str) -> str:
    m = SUFFIX_RE.search(text)
    if m:
        return text[:m.start()]
    return text


def _dedup_sentences(text: str) -> str:
    sentences = SENT_SPLIT_RE.split(text)
    seen = set()
    kept = []
    for s in sentences:
        key = s.strip()
        if len(key) < 2:
            continue
        if key in seen:
            continue
        seen.add(key)
        kept.append(key)

    # 말미에 남은 기자 약력(바이라인 소개) 문장 제거:
    # 인용부호가 있어도 화자 발화 귀속 표현(라고 했다/말했다 등)이 없다면 '진짜 인용'이 아니라고 보고
    # '-습니다'로 끝나거나 '이름+기자'가 등장하는 문장을 계속 제거한다.
    while kept:
        last = kept[-1]
        has_real_quote = any(q in last for q in QUOTE_CHARS) and bool(ATTRIBUTION_RE.search(last))
        if has_real_quote:
            break
        looks_like_bio = bool(BIO_ENDING_RE.search(last)) or bool(BIO_NAME_START_RE.match(last))
        if looks_like_bio:
            kept.pop()
            continue
        # '#해시태그 이름 기자 ...' 또는 '이름 기자 편집국 사회부' 처럼
        # 한 문장 안에 섞여 있는 기자 서명을 찾아, 서명 뒤 잔여 텍스트가 짧으면
        # (부서명 나열 등 실제 본문일 가능성이 낮음) 그 지점부터 잘라낸다.
        m = BIO_NAME_ANY_RE.search(last)
        if m and (BIO_ENDING_RE.search(last) or len(last) - m.end() <= 60):
            prefix = last[:m.start()].strip()
            kept.pop()
            if prefix:
                kept.append(prefix)
            continue
        break

    return " ".join(kept)


def _residual_cleanup(text: str) -> str:
    for pat in RESIDUAL_PATTERNS:
        text = pat.sub("", text)
    return text


def clean_article(raw_text: str) -> str:
    if not isinstance(raw_text, str) or not raw_text.strip():
        return ""
    text = raw_text.strip()
    text = _cut_prefix(text)
    text = _cut_suffix(text)
    text = _dedup_sentences(text)
    text = _residual_cleanup(text)
    text = TRAILING_HASHTAGS_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


## 3. 전체 데이터에 적용

`기사 본문 전체` 컬럼에 정제 함수를 적용해 `본문_정제` 컬럼을 새로 만듭니다.

In [ ]:
df['본문_정제'] = df['기사 본문 전체'].apply(clean_article)

df['원본_길이'] = df['기사 본문 전체'].astype(str).str.len()
df['정제_길이'] = df['본문_정제'].str.len()

print(df[['원본_길이','정제_길이']].describe())
print()
print("정제 후 200자 미만(내용이 거의 없는) 기사 수:", (df['정제_길이'] < 200).sum(), "/", len(df))
print("정제 후 0자(빈 본문) 기사 수:", (df['정제_길이'] == 0).sum())


In [ ]:
# 정제 전/후 비교 샘플 확인
import random
random.seed(0)
for i in random.sample(range(len(df)), 3):
    print("="*80)
    print("제목:", df['기사제목'].iloc[i])
    print("-"*80)
    print("[정제 전 앞부분]")
    print(str(df['기사 본문 전체'].iloc[i])[:150], "...")
    print("-"*80)
    print("[정제 후 본문]")
    print(df['본문_정제'].iloc[i])
    print()


### 품질 점검: 정제 결과가 너무 짧은 기사들 확인
아래에서 200자 미만으로 나온 기사들을 확인해보세요. 대부분은 실제로 원래 짧은 기사
(인사 발령, 사진 캡션, 날씨 정보 등)이지만, 혹시 정제 과정에서 본문이 과도하게 잘린
경우가 있는지 육안으로 확인하는 것을 권장합니다.

In [ ]:
short_df = df[df['정제_길이'] < 200].sort_values('정제_길이')
print(f"총 {len(short_df)}건")
short_df[['기사제목','정제_길이','본문_정제']].head(20)


## 4. Claude API를 이용한 2차 정제

규칙 기반 정제만으로도 대부분의 노이즈(내비게이션, 광고, 댓글, 저작권 등)는 제거되지만,
**추천기사 스니펫 일부가 문장 끝에 우연히 살아남는 경우**처럼 규칙으로 완벽히 잡기 어려운
잔여 노이즈가 드물게 남을 수 있습니다.

아래 셀은 (원하는 경우) 1차로 정제된 텍스트를 Claude에게 한 번 더 보여주고, "이 중에서
실제 기사 본문에 해당하는 문장만 남겨 달라"고 요청하는 선택적 보정 단계입니다.
- 이미 1차 정제로 텍스트가 짧아졌기 때문에(평균 15,300자 → 약 1,100자) 비용 부담이 적습니다.
- Anthropic API 키가 필요합니다 (https://console.anthropic.com 에서 발급).
- 전체 2,706건을 다 돌리기 전에, 먼저 소수 샘플로 결과를 확인.

In [ ]:
# Anthropic API 키 입력 (코랩 왼쪽 열쇠 아이콘의 '보안 비밀'에 저장해두면 더 안전합니다)
import os
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_API_KEY_HERE'  # 직접 입력할 경우

!pip install -q anthropic


In [ ]:
import anthropic

client = anthropic.Anthropic()  # 환경변수 ANTHROPIC_API_KEY 사용

REFINE_PROMPT = """다음은 뉴스 크롤링 후 1차로 규칙 기반 정제를 거친 텍스트입니다.
이미 광고/내비게이션/저작권 문구 대부분은 제거된 상태이지만, 추천기사 스니펫이나
UI 잔여 문구가 문장 끝에 조금 남아있을 수 있습니다.

아래 텍스트에서 실제 '기사 본문'에 해당하는 문장만 남기고, 그 외의 것(추천기사 제목/요약,
광고, UI 텍스트, 기자 서명 등)은 모두 제거해주세요. 본문 내용 자체는 절대 요약하거나
바꾸지 말고 원문 그대로 유지하세요. 설명이나 서두 없이 정제된 본문만 출력하세요.

--- 텍스트 시작 ---
{text}
--- 텍스트 끝 ---"""

def refine_with_claude(text: str) -> str:
    if not text or len(text) < 30:
        return text
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        messages=[{"role": "user", "content": REFINE_PROMPT.format(text=text)}],
    )
    return "".join(b.text for b in resp.content if b.type == "text").strip()

# 먼저 5건만 테스트
sample_idx = df.sample(5, random_state=1).index
for i in sample_idx:
    before = df.loc[i, '본문_정제']
    after = refine_with_claude(before)
    print("="*80)
    print("제목:", df.loc[i, '기사제목'])
    print("[1차 정제]", before[-150:])
    print("[2차 정제]", after[-150:])
    print()


In [ ]:
# 결과가 만족스러우면 전체 데이터에 적용 (시간이 다소 걸릴 수 있습니다)
# tqdm으로 진행 상황을 표시합니다.
from tqdm.auto import tqdm
tqdm.pandas()

df['본문_최종'] = df['본문_정제'].progress_apply(refine_with_claude)


## 5. 결과 저장

In [ ]:
output_cols = ['기사제목', '작성일', 'URL', '본문_정제', '검색 구분 레이블']
# 2차(LLM) 정제까지 실행했다면 '본문_최종' 컬럼을 사용하세요:
# output_cols = ['기사제목', '작성일', 'URL', '본문_최종', '검색 구분 레이블']

out = df[output_cols].rename(columns={'본문_정제': '기사 본문(정제)'})
out.to_csv('뉴스_데이터_정제결과.csv', index=False, encoding='utf-8-sig')

from google.colab import files
files.download('뉴스_데이터_정제결과.csv')
